In [1]:
# NB16_shap_three_models.ipynb

# ---------------------------------------------------------------------------
# NB16 - Full Redevelopment and SHAP Explainability for All Three Feature Sets
#
# Purpose:
#   Reviewer comments (Section 3.5.1):
#     (a) "Evaluating predictive power of override variables" - clarify whether
#         the model was fully REDEVELOPED from scratch with the expanded
#         feature set rather than forcing override variables into a fixed model.
#     (b) "Feature Importance" - provide feature-importance / SHAP analysis for
#         ALL THREE XGBoost models, not only the override-augmented one.
#
# Design:
#   Three feature sets are each used to train a SEPARATE XGBoost model from
#   scratch (independent fit, identical hyperparameters for comparability):
#     M1 financial_only     : financial ratios + raw Attr features
#     M2 financial_plus_ovr : M1 + evaluator override signals
#                             (grade_diff, override_flag) -- NO pd_system
#     M3 full_incl_pd       : M1 + full override block incl. pd_system
#   For each model we report 5-fold CV ROC-AUC (to quantify the incremental
#   value of the override block) and SHAP global importance (mean |SHAP|),
#   as native grayscale mean|SHAP| bar charts (one per model).
#
# Input:   ../data/processed/override_data.parquet
# Output:  ../results/tables/23_model_auc_by_featureset.csv
#          ../results/tables/23b_shap_meanabs_by_model.csv
#          ../results/figures/33_shap_beeswarm_financial.{png,pdf}
#          ../results/figures/34_shap_beeswarm_override.{png,pdf}
#          ../results/figures/35_shap_beeswarm_full.{png,pdf}
#          ../results/figures/35b_shap_meanabs_top.{png,pdf}
# ---------------------------------------------------------------------------

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 600, "font.family": "serif",
    "axes.edgecolor": "black", "axes.linewidth": 0.8,
    "grid.color": "0.85", "grid.linewidth": 0.5,
})
GREY_DARK, GREY_MID, GREY_LIGHT = "0.20", "0.45", "0.70"
GREYS = plt.get_cmap("Greys")

FIG_DIR, TABLE_DIR = "../results/figures/", "../results/tables/"
os.makedirs(FIG_DIR, exist_ok=True); os.makedirs(TABLE_DIR, exist_ok=True)

def savefig_current(name):
    # Save the CURRENT matplotlib figure (used for SHAP plots) as PNG + PDF.
    fig = plt.gcf()
    fig.savefig(FIG_DIR + name + ".png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG_DIR + name + ".pdf", bbox_inches="tight")
    plt.close(fig)

# ---- 1. Build the three feature sets --------------------------------------
df = pd.read_parquet("../data/processed/override_data.parquet")
FINANCIAL_COLS = ["net_profit_to_assets", "total_liabilities_to_assets",
                  "working_capital_to_assets", "current_assets_to_short_liabilities",
                  "cash_to_current_liabilities", "retained_earnings_to_assets",
                  "ebit_to_assets", "book_value_equity_to_liabilities",
                  "sales_to_assets", "equity_to_assets"]
FINANCIAL_COLS = [c for c in FINANCIAL_COLS if c in df.columns]
ATTR_COLS = [c for c in df.columns if c.startswith("Attr")]
base_feats = list(dict.fromkeys(FINANCIAL_COLS + ATTR_COLS))

FEATURE_SETS = {
    "M1_financial_only"    : base_feats,
    "M2_financial_plus_ovr": base_feats + [c for c in ["grade_diff", "override_flag"] if c in df.columns],
    "M3_full_incl_pd"      : base_feats + [c for c in ["pd_system", "grade_ordinal", "grade_diff", "override_flag"] if c in df.columns],
}
y = df["default"].astype(int)
scale_pos = (y == 0).sum() / (y == 1).sum()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def make_xgb():
    # Identical configuration for all three; each fit independently (redeveloped).
    return XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.05,
                         subsample=0.8, colsample_bytree=0.8,
                         scale_pos_weight=scale_pos, eval_metric="logloss",
                         tree_method="hist", random_state=42, n_jobs=-1, verbosity=0)

# ---- 2. CV ROC-AUC per feature set (incremental value of override block) --
auc_rows, fitted = [], {}
for name, feats in FEATURE_SETS.items():
    X = df[feats].fillna(0.0)
    scores = cross_val_score(make_xgb(), X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
    model = make_xgb().fit(X, y)                  # redeveloped from scratch
    fitted[name] = (model, X, feats)
    auc_rows.append({"model": name, "n_features": len(feats),
                     "cv_roc_auc": round(scores.mean(), 4),
                     "cv_roc_auc_std": round(scores.std(), 4)})
    print(f"{name:24s} n_feat={len(feats):3d}  CV AUC={scores.mean():.4f} +/- {scores.std():.4f}")

auc = pd.DataFrame(auc_rows)
auc["delta_vs_financial"] = (auc["cv_roc_auc"] - auc.loc[0, "cv_roc_auc"]).round(4)
auc.to_csv(TABLE_DIR + "23_model_auc_by_featureset.csv", index=False)
print("\nIncremental AUC vs financial-only:\n", auc.to_string(index=False))

# ---- 3. SHAP per model (TreeExplainer) on a common sample -----------------
# SHAP global summary shown as the standard (colour) beeswarm, one per model,
# matching the original SHAP figures in the repository. Feature value is encoded
# on SHAP's default colour map.
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(df), size=min(5000, len(df)), replace=False)
beeswarm_names = {"M1_financial_only": "33_shap_beeswarm_financial",
                  "M2_financial_plus_ovr": "34_shap_beeswarm_override",
                  "M3_full_incl_pd": "35_shap_beeswarm_full"}
meanabs_all = []
for name, (model, X, feats) in fitted.items():
    Xs = X.iloc[sample_idx]
    expl = shap.TreeExplainer(model)
    sv = expl.shap_values(Xs)
    mean_abs = np.abs(sv).mean(axis=0)
    ma = (pd.DataFrame({"model": name, "feature": feats, "mean_abs_shap": mean_abs})
          .sort_values("mean_abs_shap", ascending=False))
    meanabs_all.append(ma)
    # standard colour beeswarm (SHAP default colour map)
    plt.figure()
    shap.summary_plot(sv, Xs, max_display=12, show=False, plot_size=(7.0, 5.0))
    savefig_current(beeswarm_names[name])
    print(f"SHAP done: {name}  top feature = {ma.iloc[0]['feature']}")

meanabs = pd.concat(meanabs_all, ignore_index=True)
meanabs["mean_abs_shap"] = meanabs["mean_abs_shap"].round(6)
meanabs.to_csv(TABLE_DIR + "23b_shap_meanabs_by_model.csv", index=False)

# ---- 4. Figure 35b: top mean|SHAP| features of the full model (grayscale) -
full_top = (meanabs[meanabs["model"] == "M3_full_incl_pd"]
            .head(12).iloc[::-1])
fig, ax = plt.subplots(figsize=(6.4, 5.0))
ax.barh(full_top["feature"], full_top["mean_abs_shap"],
        color=GREY_MID, edgecolor="black", linewidth=0.6)
ax.set_xlabel("Mean |SHAP value|")
ax.set_ylabel("Feature")
fig.savefig(FIG_DIR + "35b_shap_meanabs_top.png", dpi=600, bbox_inches="tight")
fig.savefig(FIG_DIR + "35b_shap_meanabs_top.pdf", bbox_inches="tight")
plt.close(fig)

# ---- 5. Rank of override features inside the full model --------------------
m3 = meanabs[meanabs["model"] == "M3_full_incl_pd"].reset_index(drop=True)
m3["rank"] = m3["mean_abs_shap"].rank(ascending=False).astype(int)
ovr_rank = m3[m3["feature"].isin(["pd_system", "grade_ordinal", "grade_diff", "override_flag"])]
print("\nOverride-feature SHAP rank within full model:")
print(ovr_rank[["feature", "mean_abs_shap", "rank"]].to_string(index=False))
print("\nNB16 complete: per-model AUC, SHAP tables, three colour beeswarms saved.")


M1_financial_only        n_feat= 64  CV AUC=0.9639 +/- 0.0050


M2_financial_plus_ovr    n_feat= 66  CV AUC=0.9638 +/- 0.0047


M3_full_incl_pd          n_feat= 68  CV AUC=0.9628 +/- 0.0049

Incremental AUC vs financial-only:
                 model  n_features  cv_roc_auc  cv_roc_auc_std  delta_vs_financial
    M1_financial_only          64      0.9639          0.0050              0.0000
M2_financial_plus_ovr          66      0.9638          0.0047             -0.0001
      M3_full_incl_pd          68      0.9628          0.0049             -0.0011


SHAP done: M1_financial_only  top feature = Attr27


SHAP done: M2_financial_plus_ovr  top feature = Attr27


SHAP done: M3_full_incl_pd  top feature = pd_system



Override-feature SHAP rank within full model:
      feature  mean_abs_shap  rank
    pd_system       0.808071     1
grade_ordinal       0.194869     9
   grade_diff       0.030981    43
override_flag       0.004586    68

NB16 complete: per-model AUC, SHAP tables, three colour beeswarms saved.
